# IMPORTING PAKAGES

In [ ]:
#Importing pandas library
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
import tensorflow as tf

# LOADING DATASTES

In [ ]:
drive.mount('/content/drive')   

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/DSPL/DATASETS/Hotel-A-train_Cleaned_Final.csv")
df_val=pd.read_csv("/content/drive/MyDrive/Colab Notebooks/DSPL/DATASETS/Hotel-A-validation_cleaned.CSV")

In [ ]:
df.info()

In [ ]:
df_val.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.columns = df.columns.str.title()
df_val.columns = df_val.columns.str.title()
# Renaming for consistency with df
df_val = df_val.rename(columns={'Lead_Time_Days': 'Lead_Time',
                                'Stay_Duration_Nights': 'Length_Of_Stay',
                                'Visited_Previously': 'Visted_Previously'})
df_val['Reservation_Status'] = df_val['Reservation_Status'].replace('Check-In', 'Check-Out')

In [ ]:
df.isnull().sum()

In [ ]:
df_val.isnull().sum()

In [ ]:
df.Reservation_Status.value_counts()

In [ ]:
df_val.Reservation_Status.value_counts()

# FEATURE ENGINEERING

In [ ]:
# creating a new variable to calculate total number of guests per each reservation
df['Total_Guests'] = (df['Adults'] + df['Children'] + df['Babies'])
df_val['Total_Guests'] = (df_val['Adults'] + df_val['Children'] + df_val['Babies'])

In [ ]:
# creating features to capture the seasonalities
df['Expected_Checkin'] = pd.to_datetime(df['Expected_Checkin'])
df_val['Expected_Checkin'] = pd.to_datetime(df_val['Expected_Checkin'])

df['Checkin_Month'] = df['Expected_Checkin'].dt.month
df_val['Checkin_Month'] = df_val['Expected_Checkin'].dt.month

df['Booking_Date'] = pd.to_datetime(df['Booking_Date'])
df_val['Booking_Date'] = pd.to_datetime(df_val['Booking_Date'])

df['Booking_Month'] = df['Booking_Date'].dt.month
df_val['Booking_Month'] = df_val['Booking_Date'].dt.month


In [ ]:
# creating business related features
df['Actual_Room_Rate']   = df['Room_Rate'] * (1 - df['Discount_Rate'] / 100)
df_val['Actual_Room_Rate']   = df_val['Room_Rate'] * (1 - df_val['Discount_Rate'] / 100)

df['Price_Per_Guest'] = df['Actual_Room_Rate'] / df['Total_Guests']
df_val['Price_Per_Guest'] = df_val['Actual_Room_Rate'] / df_val['Total_Guests']

In [ ]:
df['Lead_Time_Category'] = pd.cut(
    df['Lead_Time'],
    # Added float('-inf') at the start
    bins=[float('-inf'),0, 7, 30, 90, 365, float('inf')],
    # Added 'Invalid/Negative' at the start to match the new bin
    labels=['post bookings', 'Last_Minute', 'Short', 'Medium', 'Long', 'VeryLong']
)
df_val['Lead_Time_Category'] = pd.cut(
    df_val['Lead_Time'],
    # Added float('-inf') at the start
    bins=[float('-inf'),0, 7, 30, 90, 365, float('inf')],
    # Added 'Invalid/Negative' at the start to match the new bin
    labels=['post bookings', 'Last_Minute', 'Short', 'Medium', 'Long', 'VeryLong']
)



# preprocessing

In [ ]:
df_val['Reservation_Status'] = df_val['Reservation_Status'].replace('Check-In', 'Check-Out')


In [ ]:
df['Reservation_Status'] = df['Reservation_Status'].str.lower()
df_val['Reservation_Status'] = df_val['Reservation_Status'].str.lower()

In [ ]:
#deleting the unwanted variables and creating Features for training data
training_feature_cols = df.drop(
['Reservation-Id','Expected_Checkin','Expected_Checkout','Booking_Date'],axis=1)

#deleting the unwanted variables and creating Features for training data
testing_feature_cols = df_val.drop(
['Reservation-Id','Expected_Checkin','Expected_Checkout','Booking_Date'],axis=1)

In [ ]:
#split dataset in features and target variable
dependent_variable='Reservation_Status' # Target variable

X_train = training_feature_cols.drop(columns=[dependent_variable])
y_train = df[dependent_variable]

X_val = testing_feature_cols.drop(columns=[dependent_variable])
y_val = df_val[dependent_variable]

In [ ]:
import pandas as pd

# Get the original column names from the feature selection step
# These are the columns that X_train and X_val should have before any transformations to ndarray
original_X_train_cols = training_feature_cols.drop(columns=[dependent_variable]).columns
original_X_val_cols = testing_feature_cols.drop(columns=[dependent_variable]).columns

# Ensure DataFrame format BEFORE pipeline
if not isinstance(X_train, pd.DataFrame):
    X_train = pd.DataFrame(X_train, columns=original_X_train_cols)

if not isinstance(X_val, pd.DataFrame):
    X_val = pd.DataFrame(X_val, columns=original_X_val_cols)

ENCODING

In [ ]:
#lable encoding
le= LabelEncoder()
y_train = le.fit_transform(y_train)

y_val = le.transform(y_val)

In [ ]:
binary_cols_train = [
    'Visted_Previously','Previous_Cancellations',
    'Use_Promotion','Required_Car_Parking'
]
X_train[binary_cols_train] = X_train[binary_cols_train].replace({'Yes':1, 'No':0}).astype(int)

binary_cols_val = [
    'Visted_Previously','Previous_Cancellations',
    'Use_Promotion','Required_Car_Parking'
]
X_val[binary_cols_val] = X_val[binary_cols_val].replace({'Yes':1, 'No':0}).astype(int)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# -----------------------------
# 1. Define column groups
# -----------------------------
one_hot_cols = ['Gender','Ethnicity','Country_Region','Hotel_Type',
                'Meal_Type','Deposit_Type','Booking_Channel']

ordinal_cols = ['Educational_Level', 'Income', 'Lead_Time_Category']

numeric_cols = [
    'Age', 'Adults', 'Children', 'Babies',
    'Length_Of_Stay', 'Visted_Previously', 'Room_Rate', 'Discount_Rate',
    'Lead_Time', 'Actual_Room_Rate', 'Price_Per_Guest', 'Total_Guests',
    'Checkin_Month', 'Booking_Month']


# Numeric pipeline
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# One-hot categorical pipeline
onehot_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Ordinal categorical pipeline
ordinal_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder())
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_cols),
    ('onehot', onehot_pipeline, one_hot_cols),
    ('ord', ordinal_pipeline, ordinal_cols)
])

# Apply pipeline
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed   = preprocessor.transform(X_val)

In [ ]:
#one-hot encoding
y_val_cat = to_categorical(y_val)
y_train_cat = to_categorical(y_train)

# MODELING


ANN MODEL

In [ ]:
def f1_macro(y_true, y_pred):
    y_pred_labels = tf.cast(tf.argmax(y_pred, axis=1), tf.int32)
    y_true_labels = tf.cast(tf.argmax(y_true, axis=1), tf.int32)
    f1_scores = []
    for c in range(3):  # one F1 per class, then average
        true_c = tf.cast(tf.equal(y_true_labels, c), tf.float32)
        pred_c = tf.cast(tf.equal(y_pred_labels, c), tf.float32)
        tp = tf.reduce_sum(true_c * pred_c)
        fp = tf.reduce_sum((1 - true_c) * pred_c)
        fn = tf.reduce_sum(true_c * (1 - pred_c))
        precision = tp / (tp + fp + 1e-8)
        recall    = tp / (tp + fn + 1e-8)
        f1 = 2 * precision * recall / (precision + recall + 1e-8)
        f1_scores.append(f1)
    return tf.reduce_mean(tf.stack(f1_scores))

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train) # Use y_train for actual class labels
class_weights_dict = dict(zip(
    classes,
    compute_class_weight(class_weight='balanced', classes=classes, y=y_train) # Pass classes and y as keyword arguments and use y_train
))

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.regularizers import l2

model = Sequential([
    Input(shape=(X_train_processed.shape[1],)),
    Dense(512, activation='relu', kernel_regularizer=l2(1e-4)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(128, activation='relu', kernel_regularizer=l2(1e-4)),
    BatchNormalization(),
    Dropout(0.2),
    Dense(64, activation='relu', kernel_regularizer=l2(1e-4)),
    Dropout(0.2),
    Dense(y_train_cat.shape[1], activation='softmax')  # dynamic output size
])

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=0.00001, clipnorm=1.0),
    loss=tf.keras.losses.CategoricalFocalCrossentropy(gamma=2.0),
    metrics=['accuracy',f1_macro]
)

In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

callbacks = [
    ReduceLROnPlateau(monitor='val_f1_macro', patience=5, factor=0.5, verbose=1, mode='max'),
    EarlyStopping(monitor='val_f1_macro', patience=15, restore_best_weights=True, mode='max')
]

In [ ]:
history = model.fit(
    X_train_processed,
    y_train_cat,
    validation_data=(X_val_processed, y_val_cat),
    epochs=100,
    batch_size=32,
    class_weight=class_weights_dict,
    callbacks=callbacks,
    verbose=1
)

In [ ]:

import numpy as np
from sklearn.metrics import classification_report, f1_score

y_pred_ann = np.argmax(model.predict(X_val_processed), axis=1)
print(classification_report(y_val, y_pred_ann, ))
print("Macro F1:", f1_score(y_val, y_pred_ann, average='macro'))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Generate confusion matrix
cm = confusion_matrix(y_val, y_pred_ann)

# Plot confusion matrix
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=le.classes_,
            yticklabels=le.classes_)

plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("ANN Confusion Matrix")
plt.show()

ANN WITH ADASYN


In [ ]:
import numpy as np
print(np.unique(y_train, return_counts=True))

In [ ]:
from imblearn.over_sampling import ADASYN
import numpy as np

adasyn = ADASYN(
    sampling_strategy={
        0: 12402,   # canceled  ( 4134, 3× original)
        2: 8500     # no-show   (2125, 4× original)
    },
    n_neighbors=5,
    random_state=42
)

X_train_resampled, y_train_resampled = adasyn.fit_resample(
    X_train_processed, y_train
)

print(np.unique(y_train_resampled, return_counts=True))

In [ ]:
#one-hot encoding
y_val_cat = to_categorical(y_val)
y_train_cat_a = to_categorical(y_train_resampled)

In [ ]:
model_adasyn = Sequential([
    Input(shape=(X_train_resampled.shape[1],)),
    Dense(256, activation='relu', kernel_regularizer=l2(1e-4)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(128, activation='relu', kernel_regularizer=l2(1e-4)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(64, activation='relu', kernel_regularizer=l2(1e-4)),
    BatchNormalization(),
    Dropout(0.2),
    Dense(3, activation='softmax')
])

In [ ]:

model_adasyn.compile(
    optimizer=Adam(learning_rate=0.00001),
    loss=tf.keras.losses.CategoricalFocalCrossentropy(gamma=2.0),
    metrics=['accuracy', f1_macro]
)

In [ ]:
callbacks_adasyn = [
    ReduceLROnPlateau(monitor='val_f1_macro', patience=5, factor=0.5, verbose=1, mode='max'),
    EarlyStopping(monitor='val_f1_macro', patience=15, restore_best_weights=True, mode='max')
]

In [ ]:
# 5. Train (no class_weight needed — SMOTE already balanced the classes)
history_adasyn = model_adasyn.fit(
    X_train_resampled, y_train_cat_a,
    validation_data=(X_val_processed, y_val_cat),
    epochs=100,
    batch_size=64,
    callbacks=callbacks_adasyn,
    verbose=1
)

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, f1_score

y_pred_ann_s = np.argmax(model_adasyn.predict(X_val_processed), axis=1)
print(classification_report(y_val, y_pred_ann_s, target_names=le.classes_))
print("Macro F1:", f1_score(y_val, y_pred_ann_s, average='macro'))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Generate confusion matrix
cm = confusion_matrix(y_val, y_pred_ann_s)

# Plot confusion matrix
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=le.classes_,
            yticklabels=le.classes_)

plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("ANN_ADASYN Confusion Matrix")
plt.show()

XGBooster

In [ ]:
# ── Fixed XGBoost — remove invalid class_weight ───────────────────────
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

xgb = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,        # fix: was 0.0001, far too slow
    max_depth=6,               # fix: was 8, reduce to avoid overfitting
    subsample=0.8,
    colsample_bytree=0.7,
    min_child_weight=3,
    gamma=0.1,
    objective='multi:softmax',
    num_class=len(classes),
    eval_metric='mlogloss',
    early_stopping_rounds=50,
    random_state=42,
    # REMOVED: class_weight='balanced'  ← invalid parameter, silently ignored
)
class_counts = np.bincount(y_train)
total = len(y_train)
sample_weight = np.array([total / (3 * class_counts[yi]) for yi in y_train])

xgb.fit(X_train_processed, y_train,
        sample_weight=sample_weight,   # ← add this
        eval_set=[(X_val_processed, y_val)],
        verbose=False)


In [ ]:
y_pred_xgb = np.argmax(xgb.predict_proba(X_val_processed), axis=1)
print(classification_report(y_val, y_pred_xgb, target_names=le.classes_))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Generate confusion matrix
cm = confusion_matrix(y_val, y_pred_xgb)

# Plot confusion matrix
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=le.classes_,
            yticklabels=le.classes_)

plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("XGBClassifier Confusion Matrix")
plt.show()

LGBMClassifier

In [ ]:
from lightgbm import LGBMClassifier
import lightgbm # Added this import
from sklearn.metrics import classification_report # Added this import

lgbm = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.02,
    class_weight='balanced',
    random_state=42
)
lgbm.fit(X_train_processed, y_train,
         eval_set=[(X_val_processed, y_val)],
         eval_metric='multi_logloss',
         callbacks=[lightgbm.early_stopping(stopping_rounds=50, verbose=False)])


In [ ]:
y_pred_lgbm = np.argmax(lgbm.predict_proba(X_val_processed), axis=1)
print(classification_report(y_val,y_pred_lgbm, target_names=le.classes_))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Generate confusion matrix
cm = confusion_matrix(y_val, y_pred_lgbm)

# Plot confusion matrix
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=le.classes_,
            yticklabels=le.classes_)

plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("LGBMClassifier Confusion Matrix")
plt.show()

# ANN WITH FEATURE SELECTION

In [ ]:
# ─────────────────────────────────────────────────────────────
# FEATURE SELECTION — run before ANN training cells
# Uses XGBoost feature importance to drop the bottom 20% of features
# ─────────────────────────────────────────────────────────────
import numpy as np
from xgboost import XGBClassifier

# ── Step 1: Train a fast XGBoost selector (no hyperparameter tuning needed) ──
selector_xgb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=5,
    objective='multi:softprob',
    num_class=3,
    eval_metric='mlogloss',
    random_state=42,
    verbosity=0
)
selector_xgb.fit(X_train_processed, y_train)

# ── Step 2: Build the feature mask ──
importances = selector_xgb.feature_importances_
threshold   = np.percentile(importances, 20)   # drop bottom 20%
selected_mask = importances >= threshold

print(f"Original features : {X_train_processed.shape[1]}")
print(f"Selected features : {selected_mask.sum()}")
print(f"Dropped features  : {(~selected_mask).sum()}")

# ── Step 3: Apply mask — use these arrays in your ANN instead ──
X_train_fs = X_train_processed[:, selected_mask]
X_val_fs   = X_val_processed[:,   selected_mask]

print(f"\nX_train_fs shape: {X_train_fs.shape}")
print(f"X_val_fs   shape: {X_val_fs.shape}")

In [ ]:
# ── Updated ANN using feature-selected inputs ──
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.regularizers import l2

model_fs = Sequential([
    Input(shape=(X_train_fs.shape[1],)),
    Dense(256, activation='relu', kernel_regularizer=l2(1e-4)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(128, activation='relu', kernel_regularizer=l2(1e-4)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(64, activation='relu', kernel_regularizer=l2(1e-4)),
    BatchNormalization(),
    Dropout(0.2),
    Dense(3, activation='softmax')
])

In [ ]:
model_fs.compile(
    optimizer=Adam(learning_rate=0.00001, clipnorm=1.0),
    loss='categorical_crossentropy',
    metrics=['accuracy',f1_macro]
)
history = model_fs.fit(
    X_train_fs,                              # <-- was X_train_processed
    y_train_cat,
    validation_data=(X_val_fs, y_val_cat),  # <-- was X_val_processed
    epochs=100,
    batch_size=64,
    class_weight=class_weights_dict,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
y_pred_ann_fs = np.argmax(model_fs.predict(X_val_fs), axis=1)   # <-- was X_val_processed

In [ ]:

print(classification_report(y_val, y_pred_ann_fs, target_names=le.classes_))
print("Macro F1:", f1_score(y_val, y_pred_ann_fs, average='macro'))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Generate confusion matrix
cm = confusion_matrix(y_val, y_pred_ann_fs)

# Plot confusion matrix
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
            xticklabels=le.classes_,
            yticklabels=le.classes_)

plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("ANN_FS Confusion Matrix")
plt.show()

# STACKING ENSEMBLE


In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.utils.class_weight import compute_class_weight
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [ ]:
class_counts = np.bincount(y_train)
total = len(y_train)
sample_weight_train = np.array([total / (3 * class_counts[yi]) for yi in y_train])

In [ ]:
def f1_macro(y_true, y_pred):
    y_pred_labels = tf.cast(tf.argmax(y_pred, axis=1), tf.int32)
    y_true_labels = tf.cast(tf.argmax(y_true, axis=1), tf.int32)
    f1_scores = []
    for c in range(3):
        true_c = tf.cast(tf.equal(y_true_labels, c), tf.float32)
        pred_c = tf.cast(tf.equal(y_pred_labels, c), tf.float32)
        tp = tf.reduce_sum(true_c * pred_c)
        fp = tf.reduce_sum((1 - true_c) * pred_c)
        fn = tf.reduce_sum(true_c * (1 - pred_c))
        p  = tp / (tp + fp + 1e-8)
        r  = tp / (tp + fn + 1e-8)
        f1_scores.append(2 * p * r / (p + r + 1e-8))
    return tf.reduce_mean(tf.stack(f1_scores))

In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight

N_FOLDS = 5
N_CLASSES = 3
cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

# Placeholder arrays: each model contributes 3 probability columns
xgb_oof  = np.zeros((len(X_train_processed), N_CLASSES))
lgbm_oof = np.zeros((len(X_train_processed), N_CLASSES))
ann_oof  = np.zeros((len(X_train_processed), N_CLASSES))

xgb_val_preds  = np.zeros((len(X_val_processed), N_CLASSES))
lgbm_val_preds = np.zeros((len(X_val_processed), N_CLASSES))
ann_val_preds  = np.zeros((len(X_val_processed), N_CLASSES))

# Precompute sample weights
class_counts = np.bincount(y_train)
total = len(y_train)
sample_weight_train = np.array([total / (3 * class_counts[yi]) for yi in y_train])

for fold, (tr_idx, vl_idx) in enumerate(cv.split(X_train_processed, y_train)):
    print(f"\n── Fold {fold+1}/{N_FOLDS} ──")
    Xtr, Xvl = X_train_processed[tr_idx], X_train_processed[vl_idx]
    ytr, yvl = y_train[tr_idx], y_train[vl_idx]
    sw = sample_weight_train[tr_idx]

    # XGBoost
    xgb_fold = XGBClassifier(
        n_estimators=600, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.7, min_child_weight=3,
        gamma=0.1, reg_alpha=0.1, reg_lambda=1.0,
        objective='multi:softprob', num_class=N_CLASSES,
        eval_metric='mlogloss', early_stopping_rounds=40,
        random_state=42, use_label_encoder=False, verbosity=0
    )
    xgb_fold.fit(Xtr, ytr, sample_weight=sw,
                 eval_set=[(Xvl, yvl)], verbose=False)
    xgb_oof[vl_idx]   = xgb_fold.predict_proba(Xvl)
    xgb_val_preds     += xgb_fold.predict_proba(X_val_processed) / N_FOLDS

    #  LightGBM
    import lightgbm as lgb
    lgbm_fold = LGBMClassifier(
        n_estimators=1000, learning_rate=0.02, num_leaves=63,
        min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0,
        class_weight='balanced', random_state=42, verbose=-1
    )
    lgbm_fold.fit(Xtr, ytr,
                  eval_set=[(Xvl, yvl)], eval_metric='multi_logloss',
                  callbacks=[lgb.early_stopping(50, verbose=False)])
    lgbm_oof[vl_idx]  = lgbm_fold.predict_proba(Xvl)
    lgbm_val_preds    += lgbm_fold.predict_proba(X_val_processed) / N_FOLDS

    #  ANN
    classes_fold = np.unique(ytr)
    cw_fold = compute_class_weight('balanced', classes=classes_fold, y=ytr)
    cw_dict = dict(zip(classes_fold, cw_fold))

    ann_fold = Sequential([
        Input(shape=(Xtr.shape[1],)),
        Dense(256, activation='relu', kernel_regularizer=l2(1e-4)),
        BatchNormalization(), Dropout(0.4),
        Dense(128, activation='relu', kernel_regularizer=l2(1e-4)),
        BatchNormalization(), Dropout(0.3),
        Dense(64,  activation='relu', kernel_regularizer=l2(1e-4)),
        BatchNormalization(), Dropout(0.2),
        Dense(N_CLASSES, activation='softmax')
    ])
    ann_fold.compile(
        optimizer=Adam(learning_rate=0.001, clipnorm=1.0),
        loss=tf.keras.losses.CategoricalFocalCrossentropy(gamma=2.0),
        metrics=[f1_macro]
    )
    ytr_cat = to_categorical(ytr, N_CLASSES)
    yvl_cat = to_categorical(yvl, N_CLASSES)
    cb_fold = [
        EarlyStopping(monitor='val_f1_macro', patience=10,
                      restore_best_weights=True, mode='max'),
        ReduceLROnPlateau(monitor='val_f1_macro', patience=5,
                          factor=0.5, mode='max', verbose=0)
    ]
    ann_fold.fit(Xtr, ytr_cat, validation_data=(Xvl, yvl_cat),
                 epochs=60, batch_size=64,
                 class_weight=cw_dict, callbacks=cb_fold, verbose=0)
    ann_oof[vl_idx]  = ann_fold.predict(Xvl, verbose=0)
    ann_val_preds   += ann_fold.predict(X_val_processed, verbose=0) / N_FOLDS
    print(f"  Fold {fold+1} complete.")

print("\nOOF generation done.")

In [ ]:
# Stack the OOF columns: 9 features total (3 models × 3 classes)
X_meta_train = np.hstack([xgb_oof, lgbm_oof, ann_oof])
X_meta_val   = np.hstack([xgb_val_preds, lgbm_val_preds, ann_val_preds])

meta_learner = LogisticRegression(
    C=1.0,
    max_iter=1000,
    class_weight='balanced',
    multi_class='multinomial',
    solver='lbfgs',
    random_state=42
)
meta_learner.fit(X_meta_train, y_train)
print("Meta-learner trained.")

In [ ]:
y_pred_stack = meta_learner.predict(X_meta_val)
print(classification_report(y_val, y_pred_stack, target_names=le.classes_))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Generate confusion matrix
cm = confusion_matrix(y_val, y_pred_stack)

# Plot confusion matrix
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=le.classes_,
            yticklabels=le.classes_)

plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("Stacking_Ensemble Confusion Matrix")
plt.show()

# HYPERPARAMETER TUNING



In [ ]:
!pip install optuna -q
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

XGBClassifier

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import f1_score

class_counts = np.bincount(y_train)
total = len(y_train)
sample_weight_train = np.array([total / (3 * class_counts[yi]) for yi in y_train])

def xgb_objective(trial):
    params = {
        'n_estimators':        trial.suggest_int('n_estimators', 300, 1000),
        'learning_rate':       trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'max_depth':           trial.suggest_int('max_depth', 3, 8),
        'subsample':           trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':    trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight':    trial.suggest_int('min_child_weight', 1, 10),
        'gamma':               trial.suggest_float('gamma', 0.0, 1.0),
        'reg_alpha':           trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':          trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'objective':           'multi:softprob',
        'num_class':           3,
        'eval_metric':         'mlogloss',
        'early_stopping_rounds': 40,
        'random_state':        42,
        'use_label_encoder':   False,
        'verbosity':           0,
    }
    m = XGBClassifier(**params)
    m.fit(X_train_processed, y_train,
          sample_weight=sample_weight_train,
          eval_set=[(X_val_processed, y_val)],
          verbose=False)
    preds = np.argmax(m.predict_proba(X_val_processed), axis=1)
    return f1_score(y_val, preds, average='macro')

study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(xgb_objective, n_trials=50, show_progress_bar=True)

print("\nBest XGBoost macro F1:", study_xgb.best_value)
print("Best XGBoost params:\n", study_xgb.best_params)

In [ ]:
# Rebuild with best params (early_stopping_rounds excluded for final model)
best_xgb_params = study_xgb.best_params.copy()

xgb_best = XGBClassifier(
    **best_xgb_params,
    objective='multi:softprob',
    num_class=3,
    eval_metric='mlogloss',
    random_state=42,
    use_label_encoder=False,
    verbosity=0
)
xgb_best.fit(X_train_processed, y_train, sample_weight=sample_weight_train, verbose=False)
print(classification_report(y_val, np.argmax(xgb_best.predict_proba(X_val_processed), axis=1), target_names=le.classes_))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Generate confusion matrix
cm = confusion_matrix(y_val, np.argmax(xgb_best.predict_proba(X_val_processed), axis=1) )

# Plot confusion matrix
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_,
            yticklabels=le.classes_)

plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("XGBClassifier_Tuned Confusion Matrix")
plt.show()

LightGBM

In [ ]:
# ── LightGBM Optuna tuning ────────────────────────────────────────────
import lightgbm as lgb
from lightgbm import LGBMClassifier

def lgbm_objective(trial):
    params = {
        'n_estimators':       trial.suggest_int('n_estimators', 300, 1500),
        'learning_rate':      trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'num_leaves':         trial.suggest_int('num_leaves', 20, 200),
        'max_depth':          trial.suggest_int('max_depth', 3, 12),
        'min_child_samples':  trial.suggest_int('min_child_samples', 10, 100),
        'subsample':          trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':   trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':          trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':         trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'class_weight':       'balanced',
        'random_state':       42,
        'verbose':            -1,
    }
    m = LGBMClassifier(**params)
    m.fit(X_train_processed, y_train,
          eval_set=[(X_val_processed, y_val)],
          eval_metric='multi_logloss',
          callbacks=[lgb.early_stopping(50, verbose=False)])
    return f1_score(y_val, m.predict(X_val_processed), average='macro')

study_lgbm = optuna.create_study(direction='maximize')
study_lgbm.optimize(lgbm_objective, n_trials=50, show_progress_bar=True)

print("\nBest LightGBM macro F1:", study_lgbm.best_value)
print("Best LightGBM params:\n", study_lgbm.best_params)

In [ ]:
lgbm_best = LGBMClassifier(
    **study_lgbm.best_params,
    class_weight='balanced',
    random_state=42,
    verbose=-1
)
lgbm_best.fit(X_train_processed, y_train)
print(classification_report(y_val, lgbm_best.predict(X_val_processed), target_names=le.classes_))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Generate confusion matrix
cm = confusion_matrix(y_val, lgbm_best.predict(X_val_processed) )

# Plot confusion matrix
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_,
            yticklabels=le.classes_)

plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("LGBMClassifier_Tuned Confusion Matrix")
plt.show()


ANN

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)
cw = compute_class_weight('balanced', classes=classes, y=y_train)
class_weights_dict = dict(zip(classes, cw))
y_train_cat = to_categorical(y_train, 3)
y_val_cat   = to_categorical(y_val,   3)

def ann_objective(trial):
    tf.random.set_seed(42)
    n_layers = trial.suggest_int('n_layers', 2,4)
    lr       = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    gamma    = trial.suggest_float('gamma', 0.5, 3.0)
    l2_reg   = trial.suggest_float('l2_reg', 1e-5, 1e-2, log=True)
    batch_sz = trial.suggest_categorical('batch_size', [32, 64, 128])

    layer_list = [Input(shape=(X_train_processed.shape[1],))]
    for i in range(n_layers):
        units   = trial.suggest_categorical(f'units_{i}', [64, 128, 256, 512])
        dropout = trial.suggest_float(f'dropout_{i}', 0.1, 0.5)
        layer_list += [
            Dense(units, activation='relu', kernel_regularizer=l2(l2_reg)),
            BatchNormalization(),
            Dropout(dropout)
        ]
    layer_list.append(Dense(3, activation='softmax'))

    model = Sequential(layer_list)
    model.compile(
        optimizer=Adam(learning_rate=lr, clipnorm=1.0),
        loss=tf.keras.losses.CategoricalFocalCrossentropy(gamma=gamma),
        metrics=[f1_macro]
    )
    cb = [
        EarlyStopping(monitor='val_f1_macro', patience=15,
                      restore_best_weights=True, mode='max'),
        ReduceLROnPlateau(monitor='val_f1_macro', patience=5,
                          factor=0.5, mode='max', verbose=0)
    ]
    model.fit(X_train_processed, y_train_cat,
              validation_data=(X_val_processed, y_val_cat),
              epochs=80, batch_size=batch_sz,
              class_weight=class_weights_dict,
              callbacks=cb, verbose=0)
    preds = np.argmax(model.predict(X_val_processed, verbose=0), axis=1)
    return f1_score(y_val, preds, average='macro')

study_ann = optuna.create_study(direction='maximize')
study_ann.optimize(ann_objective, n_trials=20, show_progress_bar=True)  # 20 trials — ANN is slow

print("\nBest ANN macro F1:", study_ann.best_value)
print("Best ANN params:\n", study_ann.best_params)

In [ ]:
best_ann = Sequential([
    Input(shape=(X_train_processed.shape[1],)),
    Dense(256, activation='relu', kernel_regularizer=l2(1.6637610254075717e-05)),
    BatchNormalization(), Dropout(0.1974353902892342),
    Dense(128, activation='relu', kernel_regularizer=l2(1.6637610254075717e-05)),
    BatchNormalization(), Dropout(0.12214765277197537),
    Dense(3, activation='softmax')
])
best_ann.compile(
    optimizer=Adam(learning_rate=0.0002932383827352342, clipnorm=1.0),
    loss=tf.keras.losses.CategoricalFocalCrossentropy(gamma=2.3491318147861087),
    metrics=[f1_macro]
)
cb_final = [
    EarlyStopping(monitor='val_f1_macro', patience=30,
                  restore_best_weights=True, mode='max'),
    ReduceLROnPlateau(monitor='val_f1_macro', patience=10,
                      factor=0.5, mode='max', verbose=1)
]
best_ann.fit(X_train_processed, y_train_cat,
             validation_data=(X_val_processed, y_val_cat),
             epochs=120, batch_size=32,
             class_weight=class_weights_dict,
             callbacks=cb_final, verbose=1)

y_pred_ann_final = np.argmax(best_ann.predict(X_val_processed, verbose=0), axis=1)
print(classification_report(y_val, y_pred_ann_final, target_names=le.classes_))
print("Macro F1:", f1_score(y_val, y_pred_ann_final, average='macro'))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Generate confusion matrix
cm = confusion_matrix(y_val,y_pred_ann_final )

# Plot confusion matrix
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_,
            yticklabels=le.classes_)

plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("ANN_Tuned Confusion Matrix")
plt.show()
